In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_8.py ──
"""
Shared infrastructure for MLFP02 Exercise 8 — FeatureStore + Feature Engineering.

Contains: HDB resale data loading, FeatureStore / ExperimentTracker setup
through kailash-ml, and OLS-from-scratch helpers reused across the four
R10 technique files:

    01_feature_schema.py        — FeatureSchema v1 + validation
    02_point_in_time.py         — Leakage prevention + temporal correctness
    03_rolling_features.py      — FeatureSchema v2 + group_by_dynamic
    04_modeling_with_features.py — Regression + hypothesis tests + Bayes

Technique-specific logic (schema construction, rolling window design,
coefficient interpretation) belongs in the per-technique files. This
module only owns infrastructure and reusable numeric helpers.
"""

from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp02_ex8"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_STORE_URL = "sqlite:///mlfp02_ex8_features.db"
FEATURE_TABLE_PREFIX = "kml_feat_"
EXPERIMENT_NAME = "mlfp02_ex8_hdb_features"


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flats (data.gov.sg)
# ════════════════════════════════════════════════════════════════════════


def load_hdb_resale() -> pl.DataFrame:
    """Load HDB resale transactions with a parsed transaction_date column.

    The raw file stores `month` as "YYYY-MM"; we convert it to a polars
    Date so every downstream technique can sort, filter, and roll on a
    real temporal axis without string parsing.
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")
    hdb = hdb.with_columns(
        pl.col("month").str.to_date("%Y-%m").alias("transaction_date")
    )
    return hdb


# ════════════════════════════════════════════════════════════════════════
# FEATURE STORE + EXPERIMENT TRACKER — kailash-ml wiring
# ════════════════════════════════════════════════════════════════════════


async def setup_feature_store() -> tuple[Any, Any, Any, bool]:
    """Create (conn, FeatureStore, ExperimentTracker, has_backend) for kailash-ml 1.1.1.

    Returns ``has_backend=False`` if the infrastructure is unavailable.
    Callers handle the degraded path by running the Polars-only versions
    of each operation.

    Note: the first tuple element is now a ``ConnectionManager`` rather than
    the old ``StoreFactory`` — kailash-ml's ExperimentTracker no longer
    accepts a positional store object; it constructs its own through the
    ``store_url`` factory. We still return a ConnectionManager so FeatureStore
    has the connection it needs.
    """
    try:
        from kailash.db import ConnectionManager
        from kailash_ml import ExperimentTracker, FeatureStore

        conn = ConnectionManager(FEATURE_STORE_URL)
        await conn.initialize()
        fs = FeatureStore(conn, table_prefix=FEATURE_TABLE_PREFIX)
        tracker = await ExperimentTracker.create(store_url=FEATURE_STORE_URL)
        return conn, fs, tracker, True
    except Exception as exc:  # noqa: BLE001 — degrade gracefully
        print(
            f"  [warn] FeatureStore backend unavailable "
            f"({type(exc).__name__}: {exc})"
        )
        return None, None, None, False


# ════════════════════════════════════════════════════════════════════════
# FEATURE COMPUTATION — v1 (basic property) and v2 (rolling market)
# ════════════════════════════════════════════════════════════════════════


def compute_v1_features(df: pl.DataFrame) -> pl.DataFrame:
    """Compute version-1 HDB property features from raw transactions.

    Produces: storey_midpoint, price_per_sqm, remaining_lease_years,
    transaction_id (row index). These are the base features v2 extends.
    """
    return df.with_columns(
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2
        ).alias("storey_midpoint"),
        (pl.col("resale_price") / pl.col("floor_area_sqm")).alias("price_per_sqm"),
        (99 - (pl.col("transaction_date").dt.year() - pl.col("lease_commence_date")))
        .cast(pl.Float64)
        .alias("remaining_lease_years"),
    ).with_row_index("transaction_id")


def compute_v2_features(df: pl.DataFrame) -> pl.DataFrame:
    """Compute v2 features = v1 + rolling town-level market context.

    Uses polars ``group_by_dynamic`` on ``transaction_date`` bucketed by
    month, then a 6-month rolling window per town. The first six months
    per town have nulls (warm-up period) — callers must ``drop_nulls``
    before modelling.
    """
    result = compute_v1_features(df).sort("transaction_date")

    town_stats = (
        result.group_by_dynamic("transaction_date", every="1mo", group_by="town")
        .agg(
            pl.col("resale_price").median().alias("monthly_median"),
            pl.col("resale_price").count().alias("monthly_volume"),
        )
        .sort("town", "transaction_date")
    )

    town_stats = town_stats.with_columns(
        pl.col("monthly_median")
        .rolling_mean(window_size=6)
        .over("town")
        .alias("town_median_price"),
        pl.col("monthly_volume")
        .rolling_sum(window_size=6)
        .over("town")
        .alias("town_transaction_volume"),
        (
            (pl.col("monthly_median") - pl.col("monthly_median").shift(6).over("town"))
            / pl.col("monthly_median").shift(6).over("town")
            * 100
        ).alias("town_price_trend"),
    )

    result = result.join(
        town_stats.select(
            "town",
            "transaction_date",
            "town_median_price",
            "town_transaction_volume",
            "town_price_trend",
        ),
        on=["town", "transaction_date"],
        how="left",
    )
    return result


# ════════════════════════════════════════════════════════════════════════
# POINT-IN-TIME RETRIEVAL HELPERS
# ════════════════════════════════════════════════════════════════════════


def as_of(
    df: pl.DataFrame, cutoff: datetime, date_col: str = "transaction_date"
) -> pl.DataFrame:
    """Return rows strictly before ``cutoff`` — the Polars-only PIT path.

    When FeatureStore is unavailable, every technique falls back to this
    helper so the leakage-prevention lesson still runs end-to-end.
    """
    return df.filter(pl.col(date_col) < pl.lit(cutoff.date()))


# ════════════════════════════════════════════════════════════════════════
# FROM-SCRATCH OLS HELPERS — reused across techniques 3 and 4
# ════════════════════════════════════════════════════════════════════════

FEATURE_LIST: list[str] = [
    "floor_area_sqm",
    "storey_midpoint",
    "remaining_lease_years",
    "town_median_price",
    "town_price_trend",
]


def prepare_design_matrix(
    df: pl.DataFrame,
    feature_list: list[str] = FEATURE_LIST,
    target: str = "resale_price",
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Drop nulls, build ``[1, X]`` design matrix, return ``(X, y, names)``."""
    model_data = df.drop_nulls(subset=[*feature_list, target])
    X_raw = model_data.select(feature_list).to_numpy().astype(np.float64)
    y = model_data[target].to_numpy().astype(np.float64)
    X = np.column_stack([np.ones(len(y)), X_raw])
    names = ["intercept", *feature_list]
    return X, y, names


def fit_ols(X: np.ndarray, y: np.ndarray) -> dict[str, Any]:
    """Fit OLS from scratch and return a dict with betas, SEs, t, p, R²."""
    from scipy import stats as sp_stats  # local import — optional at module load

    n, k = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    y_hat = X @ beta
    resid = y - y_hat

    ssr = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - ssr / sst
    adj_r2 = 1.0 - (1.0 - r2) * (n - 1) / (n - k)
    rmse = float(np.sqrt(ssr / n))

    sigma_sq = ssr / (n - k)
    xtx_inv = np.linalg.inv(X.T @ X)
    se = np.sqrt(sigma_sq * np.diag(xtx_inv))
    t_stat = beta / se
    p_val = 2.0 * (1.0 - sp_stats.t.cdf(np.abs(t_stat), df=n - k))

    sse = float(np.sum((y_hat - y.mean()) ** 2))
    f_stat = (sse / (k - 1)) / (ssr / (n - k))
    f_p = 1.0 - sp_stats.f.cdf(f_stat, dfn=k - 1, dfd=n - k)

    return {
        "n": n,
        "k": k,
        "beta": beta,
        "se": se,
        "t": t_stat,
        "p": p_val,
        "y_hat": y_hat,
        "resid": resid,
        "r2": float(r2),
        "adj_r2": float(adj_r2),
        "rmse": rmse,
        "f_stat": float(f_stat),
        "f_p": float(f_p),
    }


def normal_normal_posterior(
    beta_hat: float,
    se_hat: float,
    mu_prior: float = 0.0,
    sigma_prior: float = 10_000.0,
) -> dict[str, float]:
    """Normal-Normal conjugate posterior for a single OLS coefficient."""
    prec_prior = 1.0 / sigma_prior**2
    prec_data = 1.0 / se_hat**2
    prec_post = prec_prior + prec_data
    mu_post = (mu_prior * prec_prior + beta_hat * prec_data) / prec_post
    sigma_post = float(np.sqrt(1.0 / prec_post))
    return {
        "mu_post": float(mu_post),
        "sigma_post": sigma_post,
        "ci_low": float(mu_post - 1.96 * sigma_post),
        "ci_high": float(mu_post + 1.96 * sigma_post),
    }


# ════════════════════════════════════════════════════════════════════════
# FEATURE SCHEMA BUILDERS — kailash-ml FeatureSchema / FeatureField
# ════════════════════════════════════════════════════════════════════════


def build_schema_v1() -> Any:
    """Return the FeatureSchema v1 definition (basic property features)."""
    from kailash_ml.types import FeatureField, FeatureSchema

    return FeatureSchema(
        name="hdb_property_features",
        features=[
            FeatureField(
                name="floor_area_sqm",
                dtype="float64",
                nullable=False,
                description="Floor area in square metres",
            ),
            FeatureField(
                name="remaining_lease_years",
                dtype="float64",
                nullable=False,
                description="Remaining lease in years",
            ),
            FeatureField(
                name="storey_midpoint",
                dtype="float64",
                nullable=False,
                description="Midpoint of storey range",
            ),
            FeatureField(
                name="price_per_sqm",
                dtype="float64",
                nullable=False,
                description="Transaction price per square metre",
            ),
        ],
        entity_id_column="transaction_id",
        timestamp_column="transaction_date",
        version=1,
    )


def build_schema_v2() -> Any:
    """Return FeatureSchema v2 = v1 + three rolling market-context fields."""
    from kailash_ml.types import FeatureField, FeatureSchema

    v1 = build_schema_v1()
    return FeatureSchema(
        name="hdb_property_features",
        features=[
            *v1.features,
            FeatureField(
                name="town_median_price",
                dtype="float64",
                nullable=True,
                description="Median price in town (trailing 6 months)",
            ),
            FeatureField(
                name="town_transaction_volume",
                dtype="int64",
                nullable=True,
                description="Transaction count in town (trailing 6 months)",
            ),
            FeatureField(
                name="town_price_trend",
                dtype="float64",
                nullable=True,
                description="6-month price change % in town",
            ),
        ],
        entity_id_column="transaction_id",
        timestamp_column="transaction_date",
        version=2,
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 8.4: Regression + Lineage — Full Audit Trail
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build an OLS regression model on FeatureStore v2 features
#   - Compute from-scratch t-statistics and F-tests on coefficients
#   - Apply Normal-Normal Bayesian posteriors to interpret coefficients
#   - Log model parameters, metrics, and lineage via ExperimentTracker
#   - Generate a stakeholder report that synthesises all M2 concepts
#
# PREREQUISITES: Exercise 8.1-8.3 (schemas, PIT, rolling features)
# ESTIMATED TIME: ~50 min
#
# TASKS:
#   1. Theory — why model lineage is essential for production ML
#   2. Build — OLS regression on v2 features with full diagnostics
#   3. Train — hypothesis tests + Bayesian posteriors + ExperimentTracker
#   4. Visualise — actual vs predicted, coefficient forest, residuals
#   5. Apply — MAS regulatory model audit for Singapore banks
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as sp_stats



## THEORY — Why Model Lineage Is Essential for Production ML


In [ ]:
# A model without lineage is like a financial audit without receipts.
# You know the number on the page, but you can't prove how you got
# there. Model lineage records:
#
#   1. DATA PROVENANCE — which dataset, which version, which time range
#   2. FEATURE VERSION — which schema version produced the features
#   3. HYPERPARAMETERS — which settings were used for training
#   4. METRICS — R², RMSE, F-statistic, individual coefficient p-values
#   5. ARTIFACTS — the model weights, the training script, the config
#
# ExperimentTracker from kailash-ml captures all five automatically.
# When a regulator asks "why did your model approve this mortgage?",
# you can trace back from the prediction → model run → feature version
# → raw data → individual transaction records.
#
# Singapore context: The Monetary Authority of Singapore (MAS) requires
# banks to demonstrate model governance under FEAT principles (Fairness,
# Ethics, Accountability, Transparency). Without experiment tracking,
# banks cannot demonstrate reproducibility or explain individual
# predictions — both MAS requirements.



## TASK 2 — BUILD: OLS regression on v2 features


In [ ]:
print("\n" + "=" * 70)
print("  Exercise 8.4 — Regression + Lineage: Full Audit Trail")
print("=" * 70)

# --- 2a. Prepare v2 features ---
hdb = load_hdb_resale()
features_v2 = compute_v2_features(hdb)
n_with_market = features_v2.filter(pl.col("town_median_price").is_not_null()).height

print(f"\n  v2 features: {features_v2.shape[0]:,} rows")
print(f"  With market context: {n_with_market:,} rows")

# --- 2b. Fit OLS regression ---
# TODO: Build the design matrix from v2 features using the shared helper.
# Hint: prepare_design_matrix(features_v2) returns (X, y, names) where
# X has a column of ones prepended and names includes "intercept".
X, y, names = ____

# TODO: Fit OLS using the shared helper.
# Hint: fit_ols(X, y) returns a dict with "beta", "se", "t", "p",
# "r2", "adj_r2", "rmse", "f_stat", "f_p", "y_hat", "resid", etc.
ols = ____

print(f"\n  === Regression Model on v2 Features ===")
print(f"  n = {ols['n']:,}, k = {ols['k']}")
print(f"  R-squared = {ols['r2']:.6f} ({ols['r2']:.2%} variance explained)")
print(f"  Adj R-squared = {ols['adj_r2']:.6f}")
print(f"  RMSE = ${ols['rmse']:,.0f}")

print(f"\n  {'Feature':<25} {'Coefficient':>14}")
print("  " + "-" * 42)
for name, coef in zip(names, ols["beta"]):
    print(f"  {name:<25} {coef:>14,.2f}")



### Checkpoint 1


In [ ]:
assert ols["r2"] > 0.3, f"Task 2: R-squared should be reasonable, got {ols['r2']:.4f}"
assert ols["rmse"] > 0, "Task 2: RMSE must be positive"
print("\n[ok] Checkpoint 1 passed — regression model built on v2 features\n")

# INTERPRETATION: The R² tells us what fraction of price variation is
# explained by our 5 features. The remaining (1 - R²) is unexplained
# variance — renovation quality, unit facing, floor plan, negotiation
# skill, and luck. Adding flat-type dummies would likely improve R²
# by 5-10% (foreshadowing M3 feature selection).



## TASK 3 — TRAIN: Hypothesis tests, Bayesian posteriors, and tracking


In [ ]:
# --- 3a. Coefficient significance tests ---
print("--- Coefficient Significance (from-scratch t-statistics) ---")
print(
    f"\n  {'Feature':<25} {'beta':>12} {'SE':>10} {'t':>8} "
    f"{'p-value':>12} {'Sig':>4}"
)
print("  " + "-" * 75)
for i, name in enumerate(names):
    p = ols["p"][i]
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(
        f"  {name:<25} {ols['beta'][i]:>12,.2f} {ols['se'][i]:>10,.2f} "
        f"{ols['t'][i]:>8.2f} {p:>12.2e} {sig:>4}"
    )

print(f"\n  F-statistic: {ols['f_stat']:.2f} (p < {ols['f_p']:.2e})")
print(
    f"  Model is "
    f"{'significantly better' if ols['f_p'] < 0.05 else 'NOT better'} "
    f"than mean-only"
)

# --- 3b. Bayesian posteriors for each coefficient ---
print(f"\n--- Bayesian Posteriors (Normal-Normal Conjugate) ---")
posteriors = {}
for i, name in enumerate(names):
    if i == 0:
        continue  # Skip intercept

    # TODO: Compute the Normal-Normal posterior for this coefficient.
    # Hint: normal_normal_posterior(ols["beta"][i], ols["se"][i]) returns
    # a dict with "mu_post", "sigma_post", "ci_low", "ci_high".
    post = ____
    p_positive = 1 - sp_stats.norm.cdf(0, post["mu_post"], post["sigma_post"])
    posteriors[name] = {**post, "p_positive": p_positive}

    print(f"\n  {name}:")
    print(f"    OLS: beta={ols['beta'][i]:,.2f} +/- {ols['se'][i]:,.2f}")
    print(f"    Posterior: N({post['mu_post']:,.2f}, {post['sigma_post']:,.2f})")
    print(f"    95% credible: [{post['ci_low']:,.2f}, {post['ci_high']:,.2f}]")
    print(f"    P(beta > 0): {p_positive:.4f}")



### Checkpoint 2


In [ ]:
assert all(se > 0 for se in ols["se"]), "Task 3: standard errors must be positive"
assert len(posteriors) == len(
    FEATURE_LIST
), "Task 3: must have posteriors for all features"
print(
    "\n[ok] Checkpoint 2 passed — hypothesis tests and Bayesian posteriors complete\n"
)

# --- 3c. Log to ExperimentTracker ---
print("--- ExperimentTracker Lineage ---")

factory, fs, tracker, has_backend  = await setup_feature_store()

if has_backend:
    try:

        async def log_lineage():
            exp_id = "mlfp02_capstone_model"
            async with tracker.track(
                experiment=exp_id, run_name="hdb_price_ols_v2"
            ) as run:
                await run.log_params(
                    {
                        "feature_schema": "hdb_property_features",
                        "feature_version": "2",
                        "model_type": "OLS",
                        "n_features": str(len(FEATURE_LIST)),
                        "n_observations": str(ols["n"]),
                    }
                )
                await run.log_metrics(
                    {
                        "r2": float(ols["r2"]),
                        "adj_r2": float(ols["adj_r2"]),
                        "rmse": float(ols["rmse"]),
                        "f_statistic": float(ols["f_stat"]),
                    }
                )
                run_id = run.run_id
            return exp_id, run_id

        exp_id, run_id  = await log_lineage()

        print(f"\n  Experiment logged:")
        print(f"    Run ID: {run_id}")
        print(f"    Feature schema: hdb_property_features v2")
        print(f"    Features: {FEATURE_LIST}")
        print(f"    Training rows: {ols['n']:,}")
        print(f"    R-squared: {ols['r2']:.4f}, RMSE: ${ols['rmse']:,.0f}")
    except Exception as e:
        print(f"  [Skipped: Lineage logging ({type(e).__name__}: {e})]")
else:
    print(f"\n  [Manual lineage — ExperimentTracker unavailable]")
    print(f"    Model: OLS regression")
    print(f"    Features: {FEATURE_LIST}")
    print(f"    Training rows: {ols['n']:,}")
    print(f"    R-squared: {ols['r2']:.4f}, RMSE: ${ols['rmse']:,.0f}")



### Checkpoint 3


In [ ]:
print("\n[ok] Checkpoint 3 passed — model lineage documented\n")



## TASK 4 — VISUALISE: Actual vs predicted, coefficients, residuals


In [ ]:
print("--- Model Diagnostics Visualisations ---")

# Plot 1: Actual vs Predicted
rng = np.random.default_rng(42)
n_sample = min(3000, ols["n"])
idx = rng.choice(ols["n"], size=n_sample, replace=False)

# TODO: Create an actual vs predicted scatter plot.
# Hint: go.Scatter with x=y[idx].tolist(), y=ols["y_hat"][idx].tolist()
# then add a perfect prediction line with go.Scatter in "lines" mode.
fig1 = go.Figure()
fig1.add_trace(
    go.Scatter(
        x=____,
        y=____,
        mode="markers",
        marker={"size": 3, "opacity": 0.4, "color": "steelblue"},
        name="Predictions",
    )
)
fig1.add_trace(
    go.Scatter(
        x=[float(y.min()), float(y.max())],
        y=[float(y.min()), float(y.max())],
        mode="lines",
        name="Perfect",
        line={"dash": "dash", "color": "red"},
    )
)
fig1.update_layout(
    title=f"Capstone Model: Actual vs Predicted (R-squared={ols['r2']:.4f})",
    xaxis_title="Actual ($)",
    yaxis_title="Predicted ($)",
)
fig1.write_html(str(OUTPUT_DIR / "04_actual_vs_predicted.html"))
print(f"\n  Saved: {OUTPUT_DIR / '04_actual_vs_predicted.html'}")

# Plot 2: Coefficient forest plot
fig2 = go.Figure()
for i in range(1, ols["k"]):
    ci_lo = ols["beta"][i] - 1.96 * ols["se"][i]
    ci_hi = ols["beta"][i] + 1.96 * ols["se"][i]
    fig2.add_trace(
        go.Scatter(
            x=[ci_lo, ols["beta"][i], ci_hi],
            y=[names[i]] * 3,
            mode="markers+lines",
            name=names[i],
            marker={"size": [6, 10, 6]},
        )
    )
fig2.add_vline(x=0, line_dash="dot", line_color="red")
fig2.update_layout(
    title="Regression Coefficients with 95% CIs",
    xaxis_title="Coefficient Value",
)
fig2.write_html(str(OUTPUT_DIR / "04_coefficient_forest.html"))
print(f"  Saved: {OUTPUT_DIR / '04_coefficient_forest.html'}")

# Plot 3: Residual distribution
fig3 = go.Figure()
fig3.add_trace(
    go.Histogram(
        x=ols["resid"][idx].tolist(),
        nbinsx=50,
        marker_color="steelblue",
        opacity=0.7,
    )
)
fig3.update_layout(
    title="Residual Distribution",
    xaxis_title="Residual ($)",
    yaxis_title="Count",
)
fig3.write_html(str(OUTPUT_DIR / "04_residuals.html"))
print(f"  Saved: {OUTPUT_DIR / '04_residuals.html'}")

# TODO: Create Bayesian posterior density plots for each coefficient.
# Hint: make_subplots(rows=2, cols=3), then for each posterior,
# compute a Normal PDF using sp_stats.norm.pdf and add a go.Scatter.
fig4 = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=list(posteriors.keys()),
)
for idx_p, (name, post) in enumerate(posteriors.items()):
    row = idx_p // 3 + 1
    col = idx_p % 3 + 1
    x_range = np.linspace(
        post["ci_low"] - 2 * post["sigma_post"],
        post["ci_high"] + 2 * post["sigma_post"],
        200,
    )
    pdf_vals = sp_stats.norm.pdf(x_range, post["mu_post"], post["sigma_post"])
    fig4.add_trace(
        go.Scatter(x=x_range.tolist(), y=pdf_vals.tolist(), name=name, mode="lines"),
        row=row,
        col=col,
    )
    fig4.add_vline(x=0, line_dash="dot", line_color="red", row=row, col=col)

fig4.update_layout(title="Bayesian Posterior Densities for Coefficients", height=600)
fig4.write_html(str(OUTPUT_DIR / "04_bayesian_posteriors.html"))
print(f"  Saved: {OUTPUT_DIR / '04_bayesian_posteriors.html'}")



### Checkpoint 4


In [ ]:
print("\n[ok] Checkpoint 4 passed — all diagnostic visualisations saved\n")



## TASK 5 — APPLY: MAS Regulatory Model Audit for Singapore Banks


EXECUTIVE SUMMARY
  This analysis applies statistical methods from Module 2 to Singapore's
  HDB resale market, covering {features_v2.height:,} transactions.

  KEY FINDINGS:

  1. PRICE DISTRIBUTION (Ex 1-2: Bayesian + MLE)
     Average price: ${y.mean():,.0f} +/- ${y.std():,.0f}
     Skewness indicates {'right-skewed' if sp_stats.skew(y) > 0.5 else 'approximately symmetric'} distribution

  2. PRICE DRIVERS (Ex 5: Linear Regression)
     Our v2 model with {len(FEATURE_LIST)} features explains {ols['r2']:.1%} of


3. MARKET CONTEXT (Ex 8: Feature Engineering)
     Rolling 6-month town medians and volumes capture local market.
     {n_with_market:,} of {features_v2.height:,} transactions have context.

  4. DATA QUALITY
     Point-in-time correctness ensures no future data leaks.
     Feature versioning (v1 -> v2) tracks schema evolution.

  5. MODEL LINEAGE
     ExperimentTracker records: feature schema, model params, metrics.
     Full audit trail from raw data to final prediction.

  6. MODEL LIMITATIONS
     - {(1-ols['r2'])*100:.0f}% of variance unexplained
     - Linear model may miss non-linear relationships
     - Market features have 6-month warm-up period (nulls)

  RECOMMENDATIONS:
     - Use v2 features for all new valuation models
     - Add flat-type encoding for +5-10% R-squared improvement
     - Consider non-linear models (Random Forest, XGBoost) in M3
     - Monitor town-level trends for early price-shift signals


In [ ]:
# Scenario: The Monetary Authority of Singapore (MAS) conducts an
# annual model risk assessment of DBS Bank's property valuation model.
# MAS requires banks to demonstrate:
#   - FEAT Fairness: model doesn't discriminate by protected attributes
#   - FEAT Ethics: training data is ethically sourced (public HDB data)
#   - FEAT Accountability: full audit trail from data to prediction
#   - FEAT Transparency: model is interpretable (OLS coefficients)
#
# Without ExperimentTracker: the bank presents a PowerPoint with
# "R² = 0.76" and cannot show which data, which features, or which
# parameters produced that number. MAS flags this as a governance gap.
#
# With ExperimentTracker: the bank presents a complete lineage:
#   data version → feature schema v2 → OLS with 5 features →
#   R² = X.XX, RMSE = $Y → coefficients with p-values and CIs
# MAS approves the model governance framework.
#
# Regulatory impact: A governance gap finding can result in MAS
# requiring additional capital reserves (typically 10-20% buffer).
# For a bank with S$50B in HDB mortgage exposure, a 10% buffer
# requirement ties up S$5B in additional capital — money that cannot
# be deployed for other lending.

print("=== APPLY: MAS Regulatory Model Audit ===")
print()
print("  Scenario: MAS FEAT assessment of DBS property valuation model")
print()

# Stakeholder report
print("  " + "=" * 66)
print("  STAKEHOLDER REPORT: HDB Resale Price Analysis")
print("  " + "=" * 66)
print(
)
for i in range(1, ols["k"]):
    if ols["p"][i] < 0.05:
        print(
            f"     - {names[i]}: ${ols['beta'][i]:+,.0f} per unit "
            f"(p<{max(ols['p'][i], 1e-10):.2e})"
        )
print(
)

print("  MAS FEAT compliance summary:")
print("    Fairness:       Public HDB data, no protected-attribute features")
print("    Ethics:         Data from data.gov.sg (public, anonymised)")
print(f"    Accountability: ExperimentTracker lineage — {len(FEATURE_LIST)} features, ")
print(f"                    {ols['n']:,} rows, R-squared={ols['r2']:.4f}")
print("    Transparency:   OLS coefficients with CIs and p-values")
print()
print("  Regulatory impact of governance gap:")
print("    - 10% additional capital reserve on S$50B HDB mortgage book")
print("    - S$5B tied up in non-deployable capital")
print("    - ExperimentTracker eliminates this by providing full audit trail")



### Checkpoint 5


In [ ]:
print("\n[ok] Checkpoint 5 passed — stakeholder report and MAS audit complete\n")

# Clean up
if has_backend and factory is not None and hasattr(factory, "close"):
    try:
        asyncio.run(factory.close())
    except Exception:
        pass



## REFLECTION


[ok] OLS regression on FeatureStore v2 features
  [ok] From-scratch t-statistics and F-tests on coefficients
  [ok] Bayesian Normal-Normal posteriors for coefficient interpretation
  [ok] ExperimentTracker: parameters, metrics, and lineage logging
  [ok] Stakeholder reporting: translating statistics into business decisions
  [ok] MAS FEAT compliance: fairness, ethics, accountability, transparency

  MODULE 2 COMPLETE — YOUR STATISTICAL TOOLKIT:
  ==============================================
  Ex 1: Bayesian inference — conjugate priors, credible intervals
  Ex 2: MLE + MAP — optimisation, CLT, failure modes, AIC/BIC
  Ex 3: Hypothesis testing — bootstrap, power, BH-FDR, permutation
  Ex 4: A/B design — pre-registration, SRM, adaptive sample sizes
  Ex 5: Linear regression — OLS from scratch, VIF, WLS, diagnostics
  Ex 6: Logistic regression — sigmoid MLE, odds ratios, calibration
  Ex 7: CUPED + causal inference — variance reduction, DiD, mSPRT
  Ex 8: Capstone — feature store, lineage, complete pipeline

  -> NEXT MODULE: M3 — Supervised ML in the Kailash Pipeline
     You'll use TrainingPipeline, HyperparameterSearch, and ModelRegistry
     to build, tune, and deploy models at production scale.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

